# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/133Haseeb/MLflyrankhaseeb/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

# Ranked Actions + Reason Codes

The trained Random Forest model predicts whether a webpage is likely to require a content refresh. The predicted pages are ranked so the SEO team can review the highest-priority pages first.

| Priority | Action | Reason Code |
|----------|------------------------|-----------------------------|
| High | Refresh Immediately | LOW_CTR + OLD_CONTENT + DECLINING_TRAFFIC |
| Medium | Review Content | LOW_CTR |
| Medium | Improve SEO Elements | HIGH_IMPRESSIONS_LOW_CTR |
| Low | Monitor Performance | SMALL_DECLINE |
| Lowest | No Action | STABLE_PAGE |

The purpose of the ranking is to help SEO specialists decide which pages should be reviewed first instead of checking thousands of pages manually.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

# Intended Use

This model is designed to support SEO teams by prioritizing webpages that may require content refreshing.

The recommendations are decision-support only and should not automatically modify website content.

## Intended Users

• SEO Specialists
• Content Writers
• Content Managers

## Limits

• The model only learns from historical website data.

• It cannot understand article quality or writing style.

• It cannot detect seasonal trends or business priorities.

• It should not replace human SEO review.

• The model was trained on the available dataset and may require retraining when website behavior changes.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

# Human Review

Before refreshing a webpage, a human reviewer should check:

• Whether the content is still factually correct.

• Whether traffic changes are caused by seasonality.

• Whether Google algorithm updates affected rankings.

• Whether the page is still important for the business.

• Whether competitors changed their content.

## No-Go List

The following actions should never be fully automated:

• Publishing new content.

• Deleting webpages.

• Changing legal or medical content.

• Modifying pricing information.

• Updating business policies.

These decisions always require human approval.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

# Monitoring and Retraining

The model should be monitored regularly.

Retraining is recommended when:

• Prediction accuracy decreases.

• Large amounts of new content are added.

• Google Search behavior changes.

• Google algorithm updates significantly affect rankings.

• Website traffic patterns change.

• Every 6 to 12 months as part of regular maintenance.

Monitoring helps keep the recommendations relevant over time.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
from google.colab import files
uploaded=files.upload()

import pandas as pd
import os

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Baseline Score
df["baseline_score"] = (
    (df["days_since_last_update"] > 180).astype(int) * 40 +
    (df["ctr"] < 0.50).astype(int) * 30 +
    (df["impressions_90d"] > 5000).astype(int) * 30
)

# Recommended Action
def action(score):
    if score >= 80:
        return "Refresh Immediately"
    elif score >= 60:
        return "Review Content"
    elif score >= 40:
        return "Monitor"
    else:
        return "No Action"

df["recommended_action"] = df["baseline_score"].apply(action)

# Reason Codes
def reason(row):
    reasons = []

    if row["days_since_last_update"] > 180:
        reasons.append("OLD_CONTENT")

    if row["ctr"] < 0.50:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] > 5000:
        reasons.append("HIGH_IMPRESSIONS")

    if not reasons:
        return "STABLE_PAGE"

    return ", ".join(reasons)

df["reason_code"] = df.apply(reason, axis=1)

# Rank Pages
output = df.sort_values("baseline_score", ascending=False)

# Export
os.makedirs("work/outputs", exist_ok=True)

output.to_csv(
    "work/outputs/final_action_playbook.csv",
    index=False
)

print("Action Playbook exported successfully!")

output.head(20)

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Action Playbook exported successfully!


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,recommended_action,reason_code
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,30070.0,...,0.00,40.00,0.0,good,striking,down,-62.8,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,100,Refresh Immediately,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS"
24603,content_06f581dd9383,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,809.0,5810.0,...,0.00,100.00,0.0,low,page_1,flat,NaN,70,Review Content,"OLD_CONTENT, LOW_CTR"
6421,content_fc8cb7532683,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3913.0,27598.0,...,0.00,100.00,0.0,low,striking,down,-62.5,70,Review Content,"OLD_CONTENT, LOW_CTR"
15882,content_129753e3095f,client_19581e27de,10.0,0.55,MEDIUM,0.04,keyword article,transactional,NaN,NaN,...,0.00,100.00,0.0,low,page_3_5,down,-50.0,70,Review Content,"OLD_CONTENT, LOW_CTR"
22248,content_607ea6e3f876,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1308.0,9591.0,...,0.00,0.00,0.0,low,page_3_5,up,1080.0,70,Review Content,"OLD_CONTENT, LOW_CTR"


## Self-check

Before you submit, confirm each line honestly:

- [checked ] Every section above is filled — markdown thinking AND the code that backs it
- [checked  ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [checked  ] No client names, URLs, or private queries anywhere
- [checked  ] My claims use careful words: observed, measured, directional, decision-support
- [checked  ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.